# **4. Forecasting daily occupancy**

Chapter 3 gave us a synthetic daily history. Chapter 4 asks the practical question:
how well can we forecast the next four weeks of occupancy for a destination, and does
a smarter model beat simple rules? We build the yardstick first and the models after.

## **4.1 The forecasting task, four baselines and a ceiling**

### **4.1.1 The task**
- **Forecast:** daily occupancy for each of the five destinations, **28 days ahead**.
- **Rolling origin:** every Sunday from 2 Feb 2025 (so 13 months of history exist for
  the year-ago methods) until the last date that still has 28 days of actuals.
  At each origin a method may use only data up to and including that Sunday.
- **No leakage:** each method receives only the data up to its origin.

### **4.1.2 Four baselines (simple rules, no learning)**
1. **Last week, same weekday.** Copy the most recent matching weekday.
2. **Mean of the last 4 same weekdays.** The same idea with less noise.
3. **Last year, same weekday** (364 days earlier, so weekdays line up).
4. **Last year x recent growth.** Method 3 times the ratio of the last 28 days to the
   same 28 days a year earlier. This is the method that won in Chapter 1.

### **4.1.3 The ceiling**
An **oracle** that knows the exact generating rules but not the random noise. It is only
possible because the data are synthetic, and its error is the floor for any model: the
part of the error nobody can remove. Models will be judged on how much of the gap
between the best baseline and the oracle they close.

### **4.1.4 Metrics**
Mean absolute error in **occupancy points** (1 point = 1% of rooms), the mean error
(bias), and error by horizon band: days 1-7, 8-14 and 15-28.

### **4.1.5 Predictions (written before the run)**
1. The oracle's error is **4 to 6 points** (12% daily noise on occupancy near 0.5).
2. Every baseline lands **between 6.5 and 14 points**.
3. The **mean of the last 4 same weekdays beats last week** overall (averaging cuts noise).
4. **Last year x recent growth beats plain last year**, which should be biased low by
   about 2 to 4 points because the market grew.

### **4.1.6 Decision rule for the models in 4.3**
A model earns its place only if it (a) beats the best baseline by at least 1 point of
overall error, (b) closes at least one third of the gap between the best baseline and
the oracle, and (c) is not worse than the best baseline by more than 0.5 point in
any destination.

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import locat.forecasting.synthetic as synth

out_dir = Path("../data/processed")
DESTINATIONS = ["Maasai Mara", "Naivasha", "Diani", "Lamu", "Amboseli"]
HORIZON = 28
HISTORY_END = pd.Timestamp("2026-11-05")

# ---- data ----
syn = pd.read_csv(out_dir / "synthetic_occupancy.csv", parse_dates=["date"])
occ = syn.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]
print("history:", occ.index.min().date(), "to", occ.index.max().date(), "| days:", len(occ))

# ---- the oracle: the generator without noise, rebuilt from the saved assumptions ----
assump = json.loads((out_dir / "synthetic_assumptions.json").read_text())
params = pd.DataFrame(assump["params"])
month_adjust = pd.DataFrame.from_dict(assump["month_adjust"], orient="index")
settings = assump["globals"]
cal = pd.read_csv(out_dir / "calendar_daily.csv", parse_dates=["date"])
seasonal = pd.read_csv(out_dir / "seasonal_index.csv")
idx_map = dict(zip(seasonal["month_number"], seasonal["seasonal_index"]))

oracle_hist = synth.generate_history(
    cal, params, month_adjust, dict(settings, noise_sd=0.0), idx_map, HISTORY_END
)
oracle = oracle_hist.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]

# ---- rolling origins ----
FIRST_ORIGIN = pd.Timestamp("2025-02-02")  # a Sunday
last_origin = occ.index.max() - pd.Timedelta(days=HORIZON)
origins = pd.date_range(FIRST_ORIGIN, last_origin, freq="7D")
print(f"origins: {len(origins)} weekly | first {origins[0].date()} | last {origins[-1].date()}")


# ---- forecasting methods: each sees only `train` (data up to the origin) ----
def _dates(origin, horizon):
    return pd.date_range(origin + pd.Timedelta(days=1), periods=horizon)


def _lag7(horizon):
    """Smallest multiple of 7 days that reaches back to the last observed week."""
    return 7 * np.ceil(np.arange(1, horizon + 1) / 7).astype(int)


def fc_last_week(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    vals = [train.loc[d - pd.Timedelta(days=int(l))] for d, l in zip(dates, _lag7(horizon))]
    return pd.Series(vals, index=dates)


def fc_last4_weeks(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    vals = [
        np.mean([train.loc[d - pd.Timedelta(days=int(l) + 7 * j)] for j in range(4)])
        for d, l in zip(dates, _lag7(horizon))
    ]
    return pd.Series(vals, index=dates)


def fc_last_year(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    return pd.Series([train.loc[d - pd.Timedelta(days=364)] for d in dates], index=dates)


def fc_last_year_growth(train, origin, horizon, dest):
    base = fc_last_year(train, origin, horizon, dest)
    recent = train.loc[origin - pd.Timedelta(days=27): origin].sum()
    year_ago = train.loc[origin - pd.Timedelta(days=27 + 364): origin - pd.Timedelta(days=364)].sum()
    return base * (recent / year_ago)


def fc_oracle(train, origin, horizon, dest):
    return oracle.loc[_dates(origin, horizon), dest]


def evaluate(fn, name):
    parts = []
    for dest in DESTINATIONS:
        s = occ[dest]
        for o in origins:
            f = fn(s.loc[:o], o, HORIZON, dest)
            parts.append(pd.DataFrame({
                "method": name, "destination": dest, "origin": o,
                "h": np.arange(1, HORIZON + 1), "date": f.index,
                "forecast": f.values, "actual": s.loc[f.index].values,
            }))
    return pd.concat(parts, ignore_index=True)


methods = {
    "last_week_same_weekday": fc_last_week,
    "mean_last_4_same_weekdays": fc_last4_weeks,
    "last_year_same_weekday": fc_last_year,
    "last_year_x_recent_growth": fc_last_year_growth,
    "oracle_knows_the_rules": fc_oracle,
}
results = pd.concat([evaluate(fn, name) for name, fn in methods.items()], ignore_index=True)
results["error_pts"] = 100 * (results["forecast"] - results["actual"])
results["abs_error_pts"] = results["error_pts"].abs()
results["bucket"] = pd.cut(results["h"], [0, 7, 14, 28], labels=["days 1-7", "days 8-14", "days 15-28"])

# ---- results ----
order = list(methods)
mae = results.pivot_table(index="method", columns="destination", values="abs_error_pts", aggfunc="mean")[DESTINATIONS]
mae["ALL"] = results.groupby("method")["abs_error_pts"].mean()
print("\nMAE in occupancy points (1 point = 1% of rooms):")
print(mae.loc[order].round(1).to_string())

print("\nMean error (bias), points; negative = forecast too low:")
print(results.groupby("method")["error_pts"].mean().loc[order].round(2).to_string())

by_h = results.pivot_table(index="method", columns="bucket", values="abs_error_pts",
                           aggfunc="mean", observed=True).loc[order]
print("\nMAE by forecast horizon:")
print(by_h.round(1).to_string())

baselines = order[:-1]
best = mae.loc[baselines, "ALL"].idxmin()
print(f"\nBest baseline: {best} ({mae.loc[best, 'ALL']:.1f} pts) | "
      f"ceiling: {mae.loc['oracle_knows_the_rules', 'ALL']:.1f} pts")
print(f"rows evaluated per method: {len(results) // len(methods):,}")

results.drop(columns=["bucket"]).to_csv(out_dir / "backtest_baselines.csv", index=False)

history: 2024-01-01 to 2026-11-05 | days: 1040
origins: 88 weekly | first 2025-02-02 | last 2026-10-04

MAE in occupancy points (1 point = 1% of rooms):
destination                Maasai Mara  Naivasha  Diani  Lamu  Amboseli  ALL
method                                                                      
last_week_same_weekday             9.1       9.1   11.0   9.0       8.7  9.4
mean_last_4_same_weekdays          8.6       8.1   11.3   9.1       8.6  9.1
last_year_same_weekday             7.8       7.1    7.9   6.6       7.4  7.4
last_year_x_recent_growth          8.0       7.3    8.1   6.7       7.3  7.5
oracle_knows_the_rules             5.6       3.9    5.0   4.4       4.9  4.8

Mean error (bias), points; negative = forecast too low:
method
last_week_same_weekday      -0.14
mean_last_4_same_weekdays   -0.22
last_year_same_weekday      -2.61
last_year_x_recent_growth    0.10
oracle_knows_the_rules       0.12

MAE by forecast horizon:
bucket                     days 1-7  days 8-14  